# chamar_modelo_local.py sem Apple Silicon (alternativa multiplataforma ao MLX)

**Ahirton Lopes · Fine-Tuning Toolkit**
**Artefato de Demo - Módulo 6.2, alternativa multiplataforma ao MLX**

## O que é

O Módulo 6.2 desta disciplina expõe o checkpoint LoRA local do Módulo 4.2 como um processo
Python (`chamar_modelo_local.py`) que `amplitude-seguros-assistente.js` chama via subprocess
(`spawnSync`, flag `--local`): stdin recebe `{instrucao, entrada}` em JSON, stdout devolve o
texto bruto da resposta do modelo, mesmo contrato de retorno que `chamarModeloReal` do
Módulo 5.1. Esse script só existe em Python porque `mlx-lm` roda em Apple Silicon e não tem
biblioteca equivalente em JavaScript.

Este notebook resolve o mesmo problema pra quem não tem Mac Apple Silicon: treina o mesmo
LoRA rank 8 (mesmo dataset, mesma receita real já usada no Módulo 4.2 e no companion do
Módulo 4.2), salva o adapter em disco (o notebook original do M4.2 não salva nada, roda
tudo numa sessão só) e então recarrega esse adapter do zero pra demonstrar exatamente o
mesmo contrato stdin/stdout que `chamar_modelo_local.py` implementa, testado contra o mesmo
exemplo real (Carlos Eduardo Matos Silva) já mostrado ao vivo no vídeo do Módulo 6.2.

**MLX continua sendo o caminho oficial desta disciplina.** Este notebook, como os companions
dos Módulos 4.2 e 5.4, é caminho B, auto-contido, pra quem não tem Apple Silicon.

## Honestidade real

Este notebook já rodou de verdade numa GPU T4 do Colab, do início ao fim, sem erro - os
outputs de treino salvos abaixo (loss de treino no passo final 0,9554, loss de validação no
passo final 0,8193, acurácia média de token 79,8%) são de uma execução real, não simulada,
guardados no próprio arquivo pra quem quiser conferir sem rodar nada.


In [1]:
!pip install -q -U transformers peft trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 11.6 MB/s eta 0:00:00


In [2]:
def validar_hiperparametros(rank, max_steps, learning_rate):
    erros = []
    if not isinstance(rank, int) or not (1 <= rank <= 64):
        erros.append(f"rank fora da faixa 1-64: {rank}")
    if not isinstance(max_steps, int) or not (1 <= max_steps <= 1000):
        erros.append(f"max_steps fora da faixa 1-1000: {max_steps}")
    if not (1e-6 <= learning_rate <= 1e-2):
        erros.append(f"learning_rate fora da faixa 1e-6 a 1e-2: {learning_rate}")
    if erros:
        raise ValueError("Hiperparâmetro inválido:\n  " + "\n  ".join(erros))
    return True


RANK = 8            # mesmo posto do Módulo 4.2 real (MLX, adapter_config.json)
LORA_ALPHA = 16      # convenção comum de PEFT (2x o rank); MLX usa "scale: 20.0", conceito equivalente, escala diferente
LORA_DROPOUT = 0.0   # mesmo valor do Módulo 4.2 real
MAX_STEPS = 20       # mesmo número de iterações do Módulo 4.2 real
LEARNING_RATE = 2e-4 # convenção do próprio stack HF/PEFT - ver nota abaixo, não é comparável 1:1 com o 1e-5 do MLX
USE_DORA = False     # True reproduz a comparação DoRA do Módulo 4.3 (dora-config.yaml), com este mesmo stack

validar_hiperparametros(RANK, MAX_STEPS, LEARNING_RATE)
print("Hiperparâmetro validado.")

Hiperparâmetro validado.


**Nota real sobre o learning rate**: o `1e-5` do MLX (Módulo 4.2) e o `2e-4` deste notebook não são comparáveis número a número. Cada framework aplica o fator de escala do LoRA de um jeito diferente antes de multiplicar pelo learning rate - `scale: 20.0` no MLX, `lora_alpha: 16` aqui -, então a taxa efetiva de atualização de peso não é a mesma mesmo com o rank igual e o learning rate "parecendo" diferente por uma ordem de grandeza. Comparar val loss final entre os dois frameworks é válido; comparar o valor bruto do learning rate não é. Mesmo princípio da nota sobre `temperature` no extra do Módulo 3 (dois conceitos com o mesmo nome, escalas diferentes).

## Passo 2 - Carregar o mesmo dataset real do Módulo 2.2

157 exemplos de treino, 30 de validação, 13 de teste - os mesmos 200 exemplos reais (Amplitude Auto + Saúde Empresarial) que o Módulo 3 subiu pro Vertex AI e o Módulo 4.2 treinou local via MLX, já no formato `messages` (role/content) que os dois frameworks entendem. Embutido aqui, sem depender do resto do repositório, pra abrir e rodar sem precisar clonar nada.

In [3]:
import json

TRAIN_JSONL = r"""{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Vinicius Augusto Teixeira Placa do veiculo: YHN-3392 Data do sinistro: 20/02/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.780,90"},{"role":"assistant","content":"{\"segurado\":\"Vinicius Augusto Teixeira\",\"placa\":\"YHN-3392\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Vinicius Augusto Teixeira Placa: YHN-3392 Data do atendimento: 20/02/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 3.780,90"},{"role":"assistant","content":"{\"segurado\":\"Vinicius Augusto Teixeira\",\"placa\":\"YHN-3392\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Vinicius Augusto Teixeira Placa do veiculo: YHN-3392 Data: 20/02/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 3.780,90"},{"role":"assistant","content":"{\"segurado\":\"Vinicius Augusto Teixeira\",\"placa\":\"YHN-3392\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Vinicius Augusto Teixeira Placa do carro: YHN-3392 Data do orcamento: 20/02/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 3.780,90"},{"role":"assistant","content":"{\"segurado\":\"Vinicius Augusto Teixeira\",\"placa\":\"YHN-3392\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Vinicius Augusto Teixeira Procedimento: consulta geriatrica Data do atendimento: 20/02/2026 Valor cobrado: R$ 3.780,90"},{"role":"assistant","content":"{\"beneficiario\":\"Vinicius Augusto Teixeira\",\"procedimento\":\"consulta geriatrica\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Vinicius Augusto Teixeira Procedimento realizado: consulta geriatrica Data: 20/02/2026 Valor total: R$ 3.780,90"},{"role":"assistant","content":"{\"beneficiario\":\"Vinicius Augusto Teixeira\",\"procedimento\":\"consulta geriatrica\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Vinicius Augusto Teixeira Procedimento: consulta geriatrica Data da consulta: 20/02/2026 Valor cobrado: R$ 3.780,90"},{"role":"assistant","content":"{\"beneficiario\":\"Vinicius Augusto Teixeira\",\"procedimento\":\"consulta geriatrica\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Diego Cristina Vieira Placa do veiculo: ASD-6673 Data do sinistro: 05/04/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.870,00"},{"role":"assistant","content":"{\"segurado\":\"Diego Cristina Vieira\",\"placa\":\"ASD-6673\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Diego Cristina Vieira Placa: ASD-6673 Data do atendimento: 05/04/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 3.870,00"},{"role":"assistant","content":"{\"segurado\":\"Diego Cristina Vieira\",\"placa\":\"ASD-6673\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Diego Cristina Vieira Placa do veiculo: ASD-6673 Data: 05/04/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 3.870,00"},{"role":"assistant","content":"{\"segurado\":\"Diego Cristina Vieira\",\"placa\":\"ASD-6673\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Diego Cristina Vieira Placa do carro: ASD-6673 Data do orcamento: 05/04/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 3.870,00"},{"role":"assistant","content":"{\"segurado\":\"Diego Cristina Vieira\",\"placa\":\"ASD-6673\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Diego Cristina Vieira Placa do veiculo: ASD-6673 Data do servico: 05/04/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 3.870,00"},{"role":"assistant","content":"{\"segurado\":\"Diego Cristina Vieira\",\"placa\":\"ASD-6673\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Diego Cristina Vieira Procedimento: exame de urina completo Data do atendimento: 05/04/2026 Valor cobrado: R$ 3.870,00"},{"role":"assistant","content":"{\"beneficiario\":\"Diego Cristina Vieira\",\"procedimento\":\"exame de urina completo\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Diego Cristina Vieira Procedimento realizado: exame de urina completo Data: 05/04/2026 Valor total: R$ 3.870,00"},{"role":"assistant","content":"{\"beneficiario\":\"Diego Cristina Vieira\",\"procedimento\":\"exame de urina completo\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Diego Cristina Vieira Procedimento: exame de urina completo Data da consulta: 05/04/2026 Valor cobrado: R$ 3.870,00"},{"role":"assistant","content":"{\"beneficiario\":\"Diego Cristina Vieira\",\"procedimento\":\"exame de urina completo\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Gustavo Souza Albuquerque Placa do veiculo: TGB-2286 Data do sinistro: 10/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.590,00"},{"role":"assistant","content":"{\"segurado\":\"Gustavo Souza Albuquerque\",\"placa\":\"TGB-2286\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Gustavo Souza Albuquerque Placa: TGB-2286 Data do atendimento: 10/06/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.590,00"},{"role":"assistant","content":"{\"segurado\":\"Gustavo Souza Albuquerque\",\"placa\":\"TGB-2286\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Gustavo Souza Albuquerque Placa do veiculo: TGB-2286 Data: 10/06/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.590,00"},{"role":"assistant","content":"{\"segurado\":\"Gustavo Souza Albuquerque\",\"placa\":\"TGB-2286\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Gustavo Souza Albuquerque Placa do carro: TGB-2286 Data do orcamento: 10/06/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.590,00"},{"role":"assistant","content":"{\"segurado\":\"Gustavo Souza Albuquerque\",\"placa\":\"TGB-2286\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Gustavo Souza Albuquerque Placa do veiculo: TGB-2286 Data do servico: 10/06/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 1.590,00"},{"role":"assistant","content":"{\"segurado\":\"Gustavo Souza Albuquerque\",\"placa\":\"TGB-2286\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Gustavo Souza Albuquerque Procedimento: consulta ortopedica Data do atendimento: 10/06/2026 Valor cobrado: R$ 1.590,00"},{"role":"assistant","content":"{\"beneficiario\":\"Gustavo Souza Albuquerque\",\"procedimento\":\"consulta ortopedica\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Gustavo Souza Albuquerque Procedimento realizado: consulta ortopedica Data: 10/06/2026 Valor total: R$ 1.590,00"},{"role":"assistant","content":"{\"beneficiario\":\"Gustavo Souza Albuquerque\",\"procedimento\":\"consulta ortopedica\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Gustavo Souza Albuquerque Procedimento: consulta ortopedica Data da consulta: 10/06/2026 Valor cobrado: R$ 1.590,00"},{"role":"assistant","content":"{\"beneficiario\":\"Gustavo Souza Albuquerque\",\"procedimento\":\"consulta ortopedica\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Renata Tavares Siqueira Placa do veiculo: CDE-3396 Data do sinistro: 02/04/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.980,60"},{"role":"assistant","content":"{\"segurado\":\"Renata Tavares Siqueira\",\"placa\":\"CDE-3396\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Renata Tavares Siqueira Placa: CDE-3396 Data do atendimento: 02/04/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 3.980,60"},{"role":"assistant","content":"{\"segurado\":\"Renata Tavares Siqueira\",\"placa\":\"CDE-3396\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Renata Tavares Siqueira Placa do veiculo: CDE-3396 Data: 02/04/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 3.980,60"},{"role":"assistant","content":"{\"segurado\":\"Renata Tavares Siqueira\",\"placa\":\"CDE-3396\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Renata Tavares Siqueira Placa do carro: CDE-3396 Data do orcamento: 02/04/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 3.980,60"},{"role":"assistant","content":"{\"segurado\":\"Renata Tavares Siqueira\",\"placa\":\"CDE-3396\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Renata Tavares Siqueira Placa do veiculo: CDE-3396 Data do servico: 02/04/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 3.980,60"},{"role":"assistant","content":"{\"segurado\":\"Renata Tavares Siqueira\",\"placa\":\"CDE-3396\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Renata Tavares Siqueira Procedimento: consulta dermatologica Data do atendimento: 02/04/2026 Valor cobrado: R$ 3.980,60"},{"role":"assistant","content":"{\"beneficiario\":\"Renata Tavares Siqueira\",\"procedimento\":\"consulta dermatologica\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Renata Tavares Siqueira Procedimento realizado: consulta dermatologica Data: 02/04/2026 Valor total: R$ 3.980,60"},{"role":"assistant","content":"{\"beneficiario\":\"Renata Tavares Siqueira\",\"procedimento\":\"consulta dermatologica\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Renata Tavares Siqueira Procedimento: consulta dermatologica Data da consulta: 02/04/2026 Valor cobrado: R$ 3.980,60"},{"role":"assistant","content":"{\"beneficiario\":\"Renata Tavares Siqueira\",\"procedimento\":\"consulta dermatologica\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Bruno Luiz Bessa Placa do veiculo: PLW-7765 Data do sinistro: 16/04/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.395,00"},{"role":"assistant","content":"{\"segurado\":\"Bruno Luiz Bessa\",\"placa\":\"PLW-7765\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Bruno Luiz Bessa Placa: PLW-7765 Data do atendimento: 16/04/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.395,00"},{"role":"assistant","content":"{\"segurado\":\"Bruno Luiz Bessa\",\"placa\":\"PLW-7765\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Bruno Luiz Bessa Placa do veiculo: PLW-7765 Data: 16/04/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.395,00"},{"role":"assistant","content":"{\"segurado\":\"Bruno Luiz Bessa\",\"placa\":\"PLW-7765\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Bruno Luiz Bessa Placa do carro: PLW-7765 Data do orcamento: 16/04/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.395,00"},{"role":"assistant","content":"{\"segurado\":\"Bruno Luiz Bessa\",\"placa\":\"PLW-7765\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Bruno Luiz Bessa Placa do veiculo: PLW-7765 Data do servico: 16/04/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 1.395,00"},{"role":"assistant","content":"{\"segurado\":\"Bruno Luiz Bessa\",\"placa\":\"PLW-7765\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Bruno Luiz Bessa Procedimento: sessao de psicoterapia Data do atendimento: 16/04/2026 Valor cobrado: R$ 1.395,00"},{"role":"assistant","content":"{\"beneficiario\":\"Bruno Luiz Bessa\",\"procedimento\":\"sessao de psicoterapia\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Bruno Luiz Bessa Procedimento realizado: sessao de psicoterapia Data: 16/04/2026 Valor total: R$ 1.395,00"},{"role":"assistant","content":"{\"beneficiario\":\"Bruno Luiz Bessa\",\"procedimento\":\"sessao de psicoterapia\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Bruno Luiz Bessa Procedimento: sessao de psicoterapia Data da consulta: 16/04/2026 Valor cobrado: R$ 1.395,00"},{"role":"assistant","content":"{\"beneficiario\":\"Bruno Luiz Bessa\",\"procedimento\":\"sessao de psicoterapia\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Bruno Luiz Bessa Procedimento: sessao de psicoterapia Data do atendimento: 16/04/2026 Valor total: R$ 1.395,00"},{"role":"assistant","content":"{\"beneficiario\":\"Bruno Luiz Bessa\",\"procedimento\":\"sessao de psicoterapia\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Juliana Henrique Barros Placa do veiculo: HGF-3391 Data do sinistro: 22/07/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.950,00"},{"role":"assistant","content":"{\"segurado\":\"Juliana Henrique Barros\",\"placa\":\"HGF-3391\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Juliana Henrique Barros Placa: HGF-3391 Data do atendimento: 22/07/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 4.950,00"},{"role":"assistant","content":"{\"segurado\":\"Juliana Henrique Barros\",\"placa\":\"HGF-3391\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Juliana Henrique Barros Placa do veiculo: HGF-3391 Data: 22/07/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 4.950,00"},{"role":"assistant","content":"{\"segurado\":\"Juliana Henrique Barros\",\"placa\":\"HGF-3391\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Juliana Henrique Barros Placa do carro: HGF-3391 Data do orcamento: 22/07/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 4.950,00"},{"role":"assistant","content":"{\"segurado\":\"Juliana Henrique Barros\",\"placa\":\"HGF-3391\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Juliana Henrique Barros Placa do veiculo: HGF-3391 Data do servico: 22/07/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 4.950,00"},{"role":"assistant","content":"{\"segurado\":\"Juliana Henrique Barros\",\"placa\":\"HGF-3391\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Juliana Henrique Barros Procedimento: consulta urologica Data do atendimento: 22/07/2026 Valor cobrado: R$ 4.950,00"},{"role":"assistant","content":"{\"beneficiario\":\"Juliana Henrique Barros\",\"procedimento\":\"consulta urologica\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Juliana Henrique Barros Procedimento realizado: consulta urologica Data: 22/07/2026 Valor total: R$ 4.950,00"},{"role":"assistant","content":"{\"beneficiario\":\"Juliana Henrique Barros\",\"procedimento\":\"consulta urologica\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Juliana Henrique Barros Procedimento: consulta urologica Data da consulta: 22/07/2026 Valor cobrado: R$ 4.950,00"},{"role":"assistant","content":"{\"beneficiario\":\"Juliana Henrique Barros\",\"procedimento\":\"consulta urologica\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Juliana Henrique Barros Procedimento: consulta urologica Data do atendimento: 22/07/2026 Valor total: R$ 4.950,00"},{"role":"assistant","content":"{\"beneficiario\":\"Juliana Henrique Barros\",\"procedimento\":\"consulta urologica\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Patricia Pereira Godoy Placa do veiculo: OLP-1122 Data do sinistro: 09/04/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Patricia Pereira Godoy Placa: OLP-1122 Data do atendimento: 09/04/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Patricia Pereira Godoy Placa do veiculo: OLP-1122 Data: 09/04/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Patricia Pereira Godoy Placa do carro: OLP-1122 Data do orcamento: 09/04/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Patricia Pereira Godoy Placa do veiculo: OLP-1122 Data do servico: 09/04/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Patricia Pereira Godoy Placa: OLP-1122 Data do atendimento: 09/04/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Patricia Pereira Godoy Procedimento: consulta neurologica Data do atendimento: 09/04/2026 Valor cobrado: R$ 1.540,00"},{"role":"assistant","content":"{\"beneficiario\":\"Patricia Pereira Godoy\",\"procedimento\":\"consulta neurologica\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Patricia Pereira Godoy Procedimento realizado: consulta neurologica Data: 09/04/2026 Valor total: R$ 1.540,00"},{"role":"assistant","content":"{\"beneficiario\":\"Patricia Pereira Godoy\",\"procedimento\":\"consulta neurologica\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Patricia Pereira Godoy Procedimento: consulta neurologica Data da consulta: 09/04/2026 Valor cobrado: R$ 1.540,00"},{"role":"assistant","content":"{\"beneficiario\":\"Patricia Pereira Godoy\",\"procedimento\":\"consulta neurologica\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Patricia Pereira Godoy Procedimento: consulta neurologica Data do atendimento: 09/04/2026 Valor total: R$ 1.540,00"},{"role":"assistant","content":"{\"beneficiario\":\"Patricia Pereira Godoy\",\"procedimento\":\"consulta neurologica\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Thiago Augusto Barbosa Placa do veiculo: ERT-8873 Data do sinistro: 02/05/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Thiago Augusto Barbosa Placa: ERT-8873 Data do atendimento: 02/05/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Thiago Augusto Barbosa Placa do veiculo: ERT-8873 Data: 02/05/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Thiago Augusto Barbosa Placa do carro: ERT-8873 Data do orcamento: 02/05/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Thiago Augusto Barbosa Placa do veiculo: ERT-8873 Data do servico: 02/05/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Thiago Augusto Barbosa Placa: ERT-8873 Data do atendimento: 02/05/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Thiago Augusto Barbosa Procedimento: consulta ginecologica Data do atendimento: 02/05/2026 Valor cobrado: R$ 4.430,00"},{"role":"assistant","content":"{\"beneficiario\":\"Thiago Augusto Barbosa\",\"procedimento\":\"consulta ginecologica\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Thiago Augusto Barbosa Procedimento realizado: consulta ginecologica Data: 02/05/2026 Valor total: R$ 4.430,00"},{"role":"assistant","content":"{\"beneficiario\":\"Thiago Augusto Barbosa\",\"procedimento\":\"consulta ginecologica\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Thiago Augusto Barbosa Procedimento: consulta ginecologica Data da consulta: 02/05/2026 Valor cobrado: R$ 4.430,00"},{"role":"assistant","content":"{\"beneficiario\":\"Thiago Augusto Barbosa\",\"procedimento\":\"consulta ginecologica\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Thiago Augusto Barbosa Procedimento: consulta ginecologica Data do atendimento: 02/05/2026 Valor total: R$ 4.430,00"},{"role":"assistant","content":"{\"beneficiario\":\"Thiago Augusto Barbosa\",\"procedimento\":\"consulta ginecologica\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Beatriz Luiz Ramalho Placa do veiculo: XSW-6652 Data do sinistro: 04/02/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Beatriz Luiz Ramalho Placa: XSW-6652 Data do atendimento: 04/02/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Beatriz Luiz Ramalho Placa do veiculo: XSW-6652 Data: 04/02/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Beatriz Luiz Ramalho Placa do carro: XSW-6652 Data do orcamento: 04/02/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Beatriz Luiz Ramalho Placa do veiculo: XSW-6652 Data do servico: 04/02/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Beatriz Luiz Ramalho Placa: XSW-6652 Data do atendimento: 04/02/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Beatriz Luiz Ramalho Procedimento: exame de eletrocardiograma Data do atendimento: 04/02/2026 Valor cobrado: R$ 4.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Beatriz Luiz Ramalho\",\"procedimento\":\"exame de eletrocardiograma\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Beatriz Luiz Ramalho Procedimento realizado: exame de eletrocardiograma Data: 04/02/2026 Valor total: R$ 4.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Beatriz Luiz Ramalho\",\"procedimento\":\"exame de eletrocardiograma\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Beatriz Luiz Ramalho Procedimento: exame de eletrocardiograma Data da consulta: 04/02/2026 Valor cobrado: R$ 4.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Beatriz Luiz Ramalho\",\"procedimento\":\"exame de eletrocardiograma\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Beatriz Luiz Ramalho Procedimento: exame de eletrocardiograma Data do atendimento: 04/02/2026 Valor total: R$ 4.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Beatriz Luiz Ramalho\",\"procedimento\":\"exame de eletrocardiograma\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Beatriz Luiz Ramalho Procedimento realizado: exame de eletrocardiograma Data: 04/02/2026 Valor cobrado: R$ 4.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Beatriz Luiz Ramalho\",\"procedimento\":\"exame de eletrocardiograma\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Camila Costa Ribeiro Placa do veiculo: AZS-6617 Data do sinistro: 21/05/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Camila Costa Ribeiro Placa: AZS-6617 Data do atendimento: 21/05/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Camila Costa Ribeiro Placa do veiculo: AZS-6617 Data: 21/05/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Camila Costa Ribeiro Placa do carro: AZS-6617 Data do orcamento: 21/05/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Camila Costa Ribeiro Placa do veiculo: AZS-6617 Data do servico: 21/05/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Camila Costa Ribeiro Placa: AZS-6617 Data do atendimento: 21/05/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Camila Costa Ribeiro Procedimento: sessao de fonoterapia Data do atendimento: 21/05/2026 Valor cobrado: R$ 1.780,50"},{"role":"assistant","content":"{\"beneficiario\":\"Camila Costa Ribeiro\",\"procedimento\":\"sessao de fonoterapia\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Camila Costa Ribeiro Procedimento realizado: sessao de fonoterapia Data: 21/05/2026 Valor total: R$ 1.780,50"},{"role":"assistant","content":"{\"beneficiario\":\"Camila Costa Ribeiro\",\"procedimento\":\"sessao de fonoterapia\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Camila Costa Ribeiro Procedimento: sessao de fonoterapia Data da consulta: 21/05/2026 Valor cobrado: R$ 1.780,50"},{"role":"assistant","content":"{\"beneficiario\":\"Camila Costa Ribeiro\",\"procedimento\":\"sessao de fonoterapia\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Camila Costa Ribeiro Procedimento: sessao de fonoterapia Data do atendimento: 21/05/2026 Valor total: R$ 1.780,50"},{"role":"assistant","content":"{\"beneficiario\":\"Camila Costa Ribeiro\",\"procedimento\":\"sessao de fonoterapia\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Camila Costa Ribeiro Procedimento realizado: sessao de fonoterapia Data: 21/05/2026 Valor cobrado: R$ 1.780,50"},{"role":"assistant","content":"{\"beneficiario\":\"Camila Costa Ribeiro\",\"procedimento\":\"sessao de fonoterapia\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Eduardo Moreira Duarte Placa do veiculo: TGB-9958 Data do sinistro: 07/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Eduardo Moreira Duarte Placa: TGB-9958 Data do atendimento: 07/06/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Eduardo Moreira Duarte Placa do veiculo: TGB-9958 Data: 07/06/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Eduardo Moreira Duarte Placa do carro: TGB-9958 Data do orcamento: 07/06/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Eduardo Moreira Duarte Placa do veiculo: TGB-9958 Data do servico: 07/06/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Eduardo Moreira Duarte Placa: TGB-9958 Data do atendimento: 07/06/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Eduardo Moreira Duarte Procedimento: fisioterapia ortopedica Data do atendimento: 07/06/2026 Valor cobrado: R$ 3.660,00"},{"role":"assistant","content":"{\"beneficiario\":\"Eduardo Moreira Duarte\",\"procedimento\":\"fisioterapia ortopedica\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Eduardo Moreira Duarte Procedimento realizado: fisioterapia ortopedica Data: 07/06/2026 Valor total: R$ 3.660,00"},{"role":"assistant","content":"{\"beneficiario\":\"Eduardo Moreira Duarte\",\"procedimento\":\"fisioterapia ortopedica\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Eduardo Moreira Duarte Procedimento: fisioterapia ortopedica Data da consulta: 07/06/2026 Valor cobrado: R$ 3.660,00"},{"role":"assistant","content":"{\"beneficiario\":\"Eduardo Moreira Duarte\",\"procedimento\":\"fisioterapia ortopedica\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Eduardo Moreira Duarte Procedimento: fisioterapia ortopedica Data do atendimento: 07/06/2026 Valor total: R$ 3.660,00"},{"role":"assistant","content":"{\"beneficiario\":\"Eduardo Moreira Duarte\",\"procedimento\":\"fisioterapia ortopedica\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Eduardo Moreira Duarte Procedimento realizado: fisioterapia ortopedica Data: 07/06/2026 Valor cobrado: R$ 3.660,00"},{"role":"assistant","content":"{\"beneficiario\":\"Eduardo Moreira Duarte\",\"procedimento\":\"fisioterapia ortopedica\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Fernanda Cesar Figueiredo Placa do veiculo: POI-7738 Data do sinistro: 28/03/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Fernanda Cesar Figueiredo Placa: POI-7738 Data do atendimento: 28/03/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Fernanda Cesar Figueiredo Placa do veiculo: POI-7738 Data: 28/03/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Fernanda Cesar Figueiredo Placa do carro: POI-7738 Data do orcamento: 28/03/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Fernanda Cesar Figueiredo Placa do veiculo: POI-7738 Data do servico: 28/03/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Fernanda Cesar Figueiredo Placa: POI-7738 Data do atendimento: 28/03/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Fernanda Cesar Figueiredo Procedimento: sessao de fonoaudiologia Data do atendimento: 28/03/2026 Valor cobrado: R$ 5.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Fernanda Cesar Figueiredo\",\"procedimento\":\"sessao de fonoaudiologia\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Fernanda Cesar Figueiredo Procedimento realizado: sessao de fonoaudiologia Data: 28/03/2026 Valor total: R$ 5.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Fernanda Cesar Figueiredo\",\"procedimento\":\"sessao de fonoaudiologia\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Fernanda Cesar Figueiredo Procedimento: sessao de fonoaudiologia Data da consulta: 28/03/2026 Valor cobrado: R$ 5.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Fernanda Cesar Figueiredo\",\"procedimento\":\"sessao de fonoaudiologia\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Fernanda Cesar Figueiredo Procedimento: sessao de fonoaudiologia Data do atendimento: 28/03/2026 Valor total: R$ 5.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Fernanda Cesar Figueiredo\",\"procedimento\":\"sessao de fonoaudiologia\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Fernanda Cesar Figueiredo Procedimento realizado: sessao de fonoaudiologia Data: 28/03/2026 Valor cobrado: R$ 5.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Fernanda Cesar Figueiredo\",\"procedimento\":\"sessao de fonoaudiologia\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Joaquim Ferreira Nunes Placa do veiculo: FGH-8852 Data do sinistro: 15/07/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Joaquim Ferreira Nunes Placa: FGH-8852 Data do atendimento: 15/07/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Joaquim Ferreira Nunes Placa do veiculo: FGH-8852 Data: 15/07/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Joaquim Ferreira Nunes Placa do carro: FGH-8852 Data do orcamento: 15/07/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Joaquim Ferreira Nunes Placa do veiculo: FGH-8852 Data do servico: 15/07/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Joaquim Ferreira Nunes Placa: FGH-8852 Data do atendimento: 15/07/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Joaquim Ferreira Nunes Procedimento: sessao de acupuntura Data do atendimento: 15/07/2026 Valor cobrado: R$ 2.870,60"},{"role":"assistant","content":"{\"beneficiario\":\"Joaquim Ferreira Nunes\",\"procedimento\":\"sessao de acupuntura\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Joaquim Ferreira Nunes Procedimento realizado: sessao de acupuntura Data: 15/07/2026 Valor total: R$ 2.870,60"},{"role":"assistant","content":"{\"beneficiario\":\"Joaquim Ferreira Nunes\",\"procedimento\":\"sessao de acupuntura\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Joaquim Ferreira Nunes Procedimento: sessao de acupuntura Data da consulta: 15/07/2026 Valor cobrado: R$ 2.870,60"},{"role":"assistant","content":"{\"beneficiario\":\"Joaquim Ferreira Nunes\",\"procedimento\":\"sessao de acupuntura\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Joaquim Ferreira Nunes Procedimento: sessao de acupuntura Data do atendimento: 15/07/2026 Valor total: R$ 2.870,60"},{"role":"assistant","content":"{\"beneficiario\":\"Joaquim Ferreira Nunes\",\"procedimento\":\"sessao de acupuntura\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Joaquim Ferreira Nunes Procedimento realizado: sessao de acupuntura Data: 15/07/2026 Valor cobrado: R$ 2.870,60"},{"role":"assistant","content":"{\"beneficiario\":\"Joaquim Ferreira Nunes\",\"procedimento\":\"sessao de acupuntura\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Larissa Lopes Guimaraes Placa do veiculo: GHJ-5529 Data do sinistro: 13/02/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Larissa Lopes Guimaraes Placa: GHJ-5529 Data do atendimento: 13/02/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Larissa Lopes Guimaraes Placa do veiculo: GHJ-5529 Data: 13/02/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Larissa Lopes Guimaraes Placa do carro: GHJ-5529 Data do orcamento: 13/02/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Larissa Lopes Guimaraes Placa do veiculo: GHJ-5529 Data do servico: 13/02/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Larissa Lopes Guimaraes Placa: GHJ-5529 Data do atendimento: 13/02/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Larissa Lopes Guimaraes Procedimento: exame oftalmologico Data do atendimento: 13/02/2026 Valor cobrado: R$ 2.210,00"},{"role":"assistant","content":"{\"beneficiario\":\"Larissa Lopes Guimaraes\",\"procedimento\":\"exame oftalmologico\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Larissa Lopes Guimaraes Procedimento realizado: exame oftalmologico Data: 13/02/2026 Valor total: R$ 2.210,00"},{"role":"assistant","content":"{\"beneficiario\":\"Larissa Lopes Guimaraes\",\"procedimento\":\"exame oftalmologico\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Larissa Lopes Guimaraes Procedimento: exame oftalmologico Data da consulta: 13/02/2026 Valor cobrado: R$ 2.210,00"},{"role":"assistant","content":"{\"beneficiario\":\"Larissa Lopes Guimaraes\",\"procedimento\":\"exame oftalmologico\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Larissa Lopes Guimaraes Procedimento: exame oftalmologico Data do atendimento: 13/02/2026 Valor total: R$ 2.210,00"},{"role":"assistant","content":"{\"beneficiario\":\"Larissa Lopes Guimaraes\",\"procedimento\":\"exame oftalmologico\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Larissa Lopes Guimaraes Procedimento realizado: exame oftalmologico Data: 13/02/2026 Valor cobrado: R$ 2.210,00"},{"role":"assistant","content":"{\"beneficiario\":\"Larissa Lopes Guimaraes\",\"procedimento\":\"exame oftalmologico\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Marcos Nogueira Lima Placa do veiculo: MNB-3310 Data do sinistro: 25/03/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Marcos Nogueira Lima Placa: MNB-3310 Data do atendimento: 25/03/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Marcos Nogueira Lima Placa do veiculo: MNB-3310 Data: 25/03/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Marcos Nogueira Lima Placa do carro: MNB-3310 Data do orcamento: 25/03/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Marcos Nogueira Lima Placa do veiculo: MNB-3310 Data do servico: 25/03/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Marcos Nogueira Lima Placa: MNB-3310 Data do atendimento: 25/03/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Marcos Nogueira Lima Procedimento: exame de sangue completo Data do atendimento: 25/03/2026 Valor cobrado: R$ 4.120,30"},{"role":"assistant","content":"{\"beneficiario\":\"Marcos Nogueira Lima\",\"procedimento\":\"exame de sangue completo\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Marcos Nogueira Lima Procedimento realizado: exame de sangue completo Data: 25/03/2026 Valor total: R$ 4.120,30"},{"role":"assistant","content":"{\"beneficiario\":\"Marcos Nogueira Lima\",\"procedimento\":\"exame de sangue completo\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Marcos Nogueira Lima Procedimento: exame de sangue completo Data da consulta: 25/03/2026 Valor cobrado: R$ 4.120,30"},{"role":"assistant","content":"{\"beneficiario\":\"Marcos Nogueira Lima\",\"procedimento\":\"exame de sangue completo\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Marcos Nogueira Lima Procedimento: exame de sangue completo Data do atendimento: 25/03/2026 Valor total: R$ 4.120,30"},{"role":"assistant","content":"{\"beneficiario\":\"Marcos Nogueira Lima\",\"procedimento\":\"exame de sangue completo\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Marcos Nogueira Lima Procedimento realizado: exame de sangue completo Data: 25/03/2026 Valor cobrado: R$ 4.120,30"},{"role":"assistant","content":"{\"beneficiario\":\"Marcos Nogueira Lima\",\"procedimento\":\"exame de sangue completo\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Rafael Lopes Assuncao Placa do veiculo: CVB-1120 Data do sinistro: 12/03/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Rafael Lopes Assuncao Placa: CVB-1120 Data do atendimento: 12/03/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Rafael Lopes Assuncao Placa do veiculo: CVB-1120 Data: 12/03/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Rafael Lopes Assuncao Placa do carro: CVB-1120 Data do orcamento: 12/03/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Rafael Lopes Assuncao Placa do veiculo: CVB-1120 Data do servico: 12/03/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Rafael Lopes Assuncao Placa: CVB-1120 Data do atendimento: 12/03/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Rafael Lopes Assuncao Procedimento: exame de mamografia Data do atendimento: 12/03/2026 Valor cobrado: R$ 5.640,00"},{"role":"assistant","content":"{\"beneficiario\":\"Rafael Lopes Assuncao\",\"procedimento\":\"exame de mamografia\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Rafael Lopes Assuncao Procedimento realizado: exame de mamografia Data: 12/03/2026 Valor total: R$ 5.640,00"},{"role":"assistant","content":"{\"beneficiario\":\"Rafael Lopes Assuncao\",\"procedimento\":\"exame de mamografia\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Rafael Lopes Assuncao Procedimento: exame de mamografia Data da consulta: 12/03/2026 Valor cobrado: R$ 5.640,00"},{"role":"assistant","content":"{\"beneficiario\":\"Rafael Lopes Assuncao\",\"procedimento\":\"exame de mamografia\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Rafael Lopes Assuncao Procedimento: exame de mamografia Data do atendimento: 12/03/2026 Valor total: R$ 5.640,00"},{"role":"assistant","content":"{\"beneficiario\":\"Rafael Lopes Assuncao\",\"procedimento\":\"exame de mamografia\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Rafael Lopes Assuncao Procedimento realizado: exame de mamografia Data: 12/03/2026 Valor cobrado: R$ 5.640,00"},{"role":"assistant","content":"{\"beneficiario\":\"Rafael Lopes Assuncao\",\"procedimento\":\"exame de mamografia\",\"valor\":5640}"}]}"""

VALID_JSONL = r"""{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Rodrigo Braga Quintanilha Placa do veiculo: ZXC-2298 Data do sinistro: 29/07/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.660,40"},{"role":"assistant","content":"{\"segurado\":\"Rodrigo Braga Quintanilha\",\"placa\":\"ZXC-2298\",\"valor\":1660.4}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Rodrigo Braga Quintanilha Placa: ZXC-2298 Data do atendimento: 29/07/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.660,40"},{"role":"assistant","content":"{\"segurado\":\"Rodrigo Braga Quintanilha\",\"placa\":\"ZXC-2298\",\"valor\":1660.4}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Rodrigo Braga Quintanilha Placa do veiculo: ZXC-2298 Data: 29/07/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.660,40"},{"role":"assistant","content":"{\"segurado\":\"Rodrigo Braga Quintanilha\",\"placa\":\"ZXC-2298\",\"valor\":1660.4}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Rodrigo Braga Quintanilha Placa do carro: ZXC-2298 Data do orcamento: 29/07/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.660,40"},{"role":"assistant","content":"{\"segurado\":\"Rodrigo Braga Quintanilha\",\"placa\":\"ZXC-2298\",\"valor\":1660.4}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Rodrigo Braga Quintanilha Procedimento: sessao de terapia ocupacional Data do atendimento: 29/07/2026 Valor cobrado: R$ 1.660,40"},{"role":"assistant","content":"{\"beneficiario\":\"Rodrigo Braga Quintanilha\",\"procedimento\":\"sessao de terapia ocupacional\",\"valor\":1660.4}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Leonardo Batista Cavalcanti Placa do veiculo: YUI-2216 Data do sinistro: 30/04/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 2.220,80"},{"role":"assistant","content":"{\"segurado\":\"Leonardo Batista Cavalcanti\",\"placa\":\"YUI-2216\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Leonardo Batista Cavalcanti Placa: YUI-2216 Data do atendimento: 30/04/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 2.220,80"},{"role":"assistant","content":"{\"segurado\":\"Leonardo Batista Cavalcanti\",\"placa\":\"YUI-2216\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Leonardo Batista Cavalcanti Placa do veiculo: YUI-2216 Data: 30/04/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 2.220,80"},{"role":"assistant","content":"{\"segurado\":\"Leonardo Batista Cavalcanti\",\"placa\":\"YUI-2216\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Leonardo Batista Cavalcanti Placa do carro: YUI-2216 Data do orcamento: 30/04/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 2.220,80"},{"role":"assistant","content":"{\"segurado\":\"Leonardo Batista Cavalcanti\",\"placa\":\"YUI-2216\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Leonardo Batista Cavalcanti Procedimento: exame de imagem (ressonancia) Data do atendimento: 30/04/2026 Valor cobrado: R$ 2.220,80"},{"role":"assistant","content":"{\"beneficiario\":\"Leonardo Batista Cavalcanti\",\"procedimento\":\"exame de imagem (ressonancia)\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Leonardo Batista Cavalcanti Procedimento realizado: exame de imagem (ressonancia) Data: 30/04/2026 Valor total: R$ 2.220,80"},{"role":"assistant","content":"{\"beneficiario\":\"Leonardo Batista Cavalcanti\",\"procedimento\":\"exame de imagem (ressonancia)\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Priscila Regina Coutinho Placa do veiculo: WSX-6656 Data do sinistro: 03/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.310,00"},{"role":"assistant","content":"{\"segurado\":\"Priscila Regina Coutinho\",\"placa\":\"WSX-6656\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Priscila Regina Coutinho Placa: WSX-6656 Data do atendimento: 03/06/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 4.310,00"},{"role":"assistant","content":"{\"segurado\":\"Priscila Regina Coutinho\",\"placa\":\"WSX-6656\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Priscila Regina Coutinho Placa do veiculo: WSX-6656 Data: 03/06/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 4.310,00"},{"role":"assistant","content":"{\"segurado\":\"Priscila Regina Coutinho\",\"placa\":\"WSX-6656\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Priscila Regina Coutinho Placa do carro: WSX-6656 Data do orcamento: 03/06/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 4.310,00"},{"role":"assistant","content":"{\"segurado\":\"Priscila Regina Coutinho\",\"placa\":\"WSX-6656\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Priscila Regina Coutinho Procedimento: exame de densitometria ossea Data do atendimento: 03/06/2026 Valor cobrado: R$ 4.310,00"},{"role":"assistant","content":"{\"beneficiario\":\"Priscila Regina Coutinho\",\"procedimento\":\"exame de densitometria ossea\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Priscila Regina Coutinho Procedimento realizado: exame de densitometria ossea Data: 03/06/2026 Valor total: R$ 4.310,00"},{"role":"assistant","content":"{\"beneficiario\":\"Priscila Regina Coutinho\",\"procedimento\":\"exame de densitometria ossea\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Sabrina Rocha Pimentel Placa do veiculo: FDS-8842 Data do sinistro: 08/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 890,00"},{"role":"assistant","content":"{\"segurado\":\"Sabrina Rocha Pimentel\",\"placa\":\"FDS-8842\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Sabrina Rocha Pimentel Placa: FDS-8842 Data do atendimento: 08/06/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 890,00"},{"role":"assistant","content":"{\"segurado\":\"Sabrina Rocha Pimentel\",\"placa\":\"FDS-8842\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Sabrina Rocha Pimentel Placa do veiculo: FDS-8842 Data: 08/06/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 890,00"},{"role":"assistant","content":"{\"segurado\":\"Sabrina Rocha Pimentel\",\"placa\":\"FDS-8842\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Sabrina Rocha Pimentel Placa do carro: FDS-8842 Data do orcamento: 08/06/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 890,00"},{"role":"assistant","content":"{\"segurado\":\"Sabrina Rocha Pimentel\",\"placa\":\"FDS-8842\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Sabrina Rocha Pimentel Procedimento: exame de tomografia Data do atendimento: 08/06/2026 Valor cobrado: R$ 890,00"},{"role":"assistant","content":"{\"beneficiario\":\"Sabrina Rocha Pimentel\",\"procedimento\":\"exame de tomografia\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Sabrina Rocha Pimentel Procedimento realizado: exame de tomografia Data: 08/06/2026 Valor total: R$ 890,00"},{"role":"assistant","content":"{\"beneficiario\":\"Sabrina Rocha Pimentel\",\"procedimento\":\"exame de tomografia\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Mariana dos Santos Pena Placa do veiculo: QWE-1150 Data do sinistro: 24/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.485,70"},{"role":"assistant","content":"{\"segurado\":\"Mariana dos Santos Pena\",\"placa\":\"QWE-1150\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Mariana dos Santos Pena Placa: QWE-1150 Data do atendimento: 24/06/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.485,70"},{"role":"assistant","content":"{\"segurado\":\"Mariana dos Santos Pena\",\"placa\":\"QWE-1150\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Mariana dos Santos Pena Placa do veiculo: QWE-1150 Data: 24/06/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.485,70"},{"role":"assistant","content":"{\"segurado\":\"Mariana dos Santos Pena\",\"placa\":\"QWE-1150\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Mariana dos Santos Pena Placa do carro: QWE-1150 Data do orcamento: 24/06/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.485,70"},{"role":"assistant","content":"{\"segurado\":\"Mariana dos Santos Pena\",\"placa\":\"QWE-1150\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Mariana dos Santos Pena Procedimento: sessao de pilates terapeutico Data do atendimento: 24/06/2026 Valor cobrado: R$ 1.485,70"},{"role":"assistant","content":"{\"beneficiario\":\"Mariana dos Santos Pena\",\"procedimento\":\"sessao de pilates terapeutico\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Mariana dos Santos Pena Procedimento realizado: sessao de pilates terapeutico Data: 24/06/2026 Valor total: R$ 1.485,70"},{"role":"assistant","content":"{\"beneficiario\":\"Mariana dos Santos Pena\",\"procedimento\":\"sessao de pilates terapeutico\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Mariana dos Santos Pena Procedimento: sessao de pilates terapeutico Data da consulta: 24/06/2026 Valor cobrado: R$ 1.485,70"},{"role":"assistant","content":"{\"beneficiario\":\"Mariana dos Santos Pena\",\"procedimento\":\"sessao de pilates terapeutico\",\"valor\":1485.7}"}]}"""

TEST_JSONL = r"""{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Anderson Machado Freitas Placa do veiculo: WER-4481 Data do sinistro: 01/07/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.390,60"},{"role":"assistant","content":"{\"segurado\":\"Anderson Machado Freitas\",\"placa\":\"WER-4481\",\"valor\":3390.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Debora Martins Cardoso Placa do veiculo: LKM-3376 Data do sinistro: 14/03/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.780,60"},{"role":"assistant","content":"{\"segurado\":\"Debora Martins Cardoso\",\"placa\":\"LKM-3376\",\"valor\":4780.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Fabio Vinicius Andrade Pereira Placa do veiculo: UJM-7726 Data do sinistro: 23/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 6.310,90"},{"role":"assistant","content":"{\"segurado\":\"Fabio Vinicius Andrade Pereira\",\"placa\":\"UJM-7726\",\"valor\":6310.9}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Vanessa Costa Miranda Placa do veiculo: DFG-7784 Data do sinistro: 27/02/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 2.150,80"},{"role":"assistant","content":"{\"segurado\":\"Vanessa Costa Miranda\",\"placa\":\"DFG-7784\",\"valor\":2150.8}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Carolina Almeida Correia Placa do veiculo: QJK-4F82 Data do sinistro: 11/05/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 2.510,30"},{"role":"assistant","content":"{\"segurado\":\"Carolina Almeida Correia\",\"placa\":\"QJK-4F82\",\"valor\":2510.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Carolina Almeida Correia Placa: QJK-4F82 Data do atendimento: 11/05/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 2.510,30"},{"role":"assistant","content":"{\"segurado\":\"Carolina Almeida Correia\",\"placa\":\"QJK-4F82\",\"valor\":2510.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Felipe Alves Monteiro Placa do veiculo: YHN-5520 Data do sinistro: 18/05/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.450,00"},{"role":"assistant","content":"{\"segurado\":\"Felipe Alves Monteiro\",\"placa\":\"YHN-5520\",\"valor\":3450}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Felipe Alves Monteiro Placa: YHN-5520 Data do atendimento: 18/05/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 3.450,00"},{"role":"assistant","content":"{\"segurado\":\"Felipe Alves Monteiro\",\"placa\":\"YHN-5520\",\"valor\":3450}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Felipe Alves Monteiro Procedimento: consulta de clinica geral Data do atendimento: 18/05/2026 Valor cobrado: R$ 3.450,00"},{"role":"assistant","content":"{\"beneficiario\":\"Felipe Alves Monteiro\",\"procedimento\":\"consulta de clinica geral\",\"valor\":3450}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Amanda Pedro Salgado Placa do veiculo: MJU-6624 Data do sinistro: 17/03/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.870,00"},{"role":"assistant","content":"{\"segurado\":\"Amanda Pedro Salgado\",\"placa\":\"MJU-6624\",\"valor\":1870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Amanda Pedro Salgado Placa: MJU-6624 Data do atendimento: 17/03/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.870,00"},{"role":"assistant","content":"{\"segurado\":\"Amanda Pedro Salgado\",\"placa\":\"MJU-6624\",\"valor\":1870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Amanda Pedro Salgado Placa do veiculo: MJU-6624 Data: 17/03/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.870,00"},{"role":"assistant","content":"{\"segurado\":\"Amanda Pedro Salgado\",\"placa\":\"MJU-6624\",\"valor\":1870}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Amanda Pedro Salgado Procedimento: exame de audiometria Data do atendimento: 17/03/2026 Valor cobrado: R$ 1.870,00"},{"role":"assistant","content":"{\"beneficiario\":\"Amanda Pedro Salgado\",\"procedimento\":\"exame de audiometria\",\"valor\":1870}"}]}"""

def carregar(jsonl_text):
    return [json.loads(linha) for linha in jsonl_text.strip().splitlines()]

exemplos_treino = carregar(TRAIN_JSONL)
exemplos_validacao = carregar(VALID_JSONL)
exemplos_teste = carregar(TEST_JSONL)

print(f"Treino: {len(exemplos_treino)} exemplos")
print(f"Validação: {len(exemplos_validacao)} exemplos")
print(f"Teste: {len(exemplos_teste)} exemplos")
assert len(exemplos_treino) == 157 and len(exemplos_validacao) == 30 and len(exemplos_teste) == 13
print(exemplos_treino[0])

Treino: 157 exemplos
Validação: 30 exemplos
Teste: 13 exemplos
{'messages': [{'role': 'user', 'content': 'Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Vinicius Augusto Teixeira Placa do veiculo: YHN-3392 Data do sinistro: 20/02/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.780,90'}, {'role': 'assistant', 'content': '{"segurado":"Vinicius Augusto Teixeira","placa":"YHN-3392","valor":3780.9}'}]}


## Passo 3 - Carregar o modelo em 4 bits (QLoRA) e configurar o LoRA

Segue o padrão oficial do próprio Google pra fine-tuning de Gemma via Hugging Face (`AutoModelForMultimodalLM` + `AutoProcessor`, mesmo pra uso só de texto: o Gemma 4 é nativamente multimodal). Quantização em 4 bits (`BitsAndBytesConfig`) é o que faz um modelo de ~4,63B parâmetros caber na GPU T4 grátis do Colab (16GB de VRAM).

**Achado real rodando isso pela primeira vez**: `prepare_model_for_kbit_training` tentou alocar 8,75 GiB numa tacada só e estourou a memória da T4, mesmo com quantização em 4 bits ativa. Diagnóstico real (célula abaixo) apontou a causa exata: `embed_tokens_per_layer.weight`, 2,35 bilhões de elementos - a tabela de "per-layer embeddings" que dá nome ao "E2B" (parâmetro bruto alto, computação efetiva baixa, porque é consulta, não multiplicação de matriz). `prepare_model_for_kbit_training` promove todo parâmetro não-4bit pra float32 sem exceção de tamanho, e essa tabela sozinha já pede 9,4GB. Ela fica congelada de qualquer jeito (LoRA não treina embedding), então não precisava desse upcast nunca - a célula abaixo reimplementa a mesma função pulando essa promoção acima de um limiar de tamanho.

In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Precisa ser setado ANTES do primeiro import de torch/CUDA pra ter efeito - por isso é a primeira linha
# desta célula. Fix sugerido pelo próprio erro do PyTorch: memória "livre" reportada nem sempre é
# contígua o suficiente pra uma alocação grande numa tacada só; isso deixa o alocador crescer em blocos.

import gc
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig

MODEL_ID = "google/gemma-4-E2B-it"  # mesmo modelo do Módulo 4.2 (lá: mlx-community/gemma-4-e2b-it-bf16)

torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_quant_storage=torch_dtype,
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch_dtype,
    device_map="auto",
    quantization_config=quantization_config,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

gc.collect()
torch.cuda.empty_cache()
print(f"Memória GPU alocada antes do prepare_model_for_kbit_training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Diagnóstico: se ainda estourar memória aqui embaixo, esta lista mostra qual parâmetro específico
# não ficou em 4 bits e é grande o suficiente pra explicar o estouro - útil pra reportar de volta.
maiores_nao_quantizados = sorted(
    (
        (nome, p.numel(), p.dtype)
        for nome, p in model.named_parameters()
        if p.__class__.__name__ != "Params4bit" and p.dtype in (torch.float16, torch.bfloat16)
    ),
    key=lambda item: -item[1],
)[:5]
print("Maiores parâmetros ainda em 16 bits (candidatos ao upcast que pode estourar memória):")
for nome, numel, dtype in maiores_nao_quantizados:
    gb_em_fp32 = numel * 4 / 1e9
    print(f"  {nome}: {numel:,} elementos, {dtype}, ~{gb_em_fp32:.2f} GB se virar float32")

LIMIAR_ELEMENTOS_UPCAST = 50_000_000
# Achado real, rodando isso pela primeira vez: o Gemma 4 usa "per-layer embeddings" (é o que o "E2B"
# do nome quer dizer - parâmetro bruto alto, custo de computação baixo, porque é consulta, não
# multiplicação de matriz). Isso cria um parâmetro, embed_tokens_per_layer, com 2,35 bilhões de
# elementos - e peft.prepare_model_for_kbit_training tenta promover TODO parâmetro não-4bit pra
# float32, sem exceção de tamanho, o que sozinho já pede 9,4GB numa GPU de 14,56GB. Essa promoção
# nunca precisaria valer pra essa tabela: ela fica congelada (linha abaixo replica esse
# congelamento) e LoRA não treina embedding (target_modules="all-linear" só afeta camada linear).
# Reimplementação abaixo é idêntica à função original do peft (conferida no código-fonte da
# biblioteca), com uma única mudança: pula o upcast em parâmetro acima do limiar.


def preparar_modelo_para_treino_kbit(modelo, gradient_checkpointing_kwargs=None):
    if gradient_checkpointing_kwargs is None:
        gradient_checkpointing_kwargs = {}

    for param in modelo.parameters():
        param.requires_grad = False

    for param in modelo.parameters():
        eh_fp16_ou_bf16 = param.dtype in (torch.float16, torch.bfloat16)
        eh_grande_demais = param.numel() > LIMIAR_ELEMENTOS_UPCAST
        if eh_fp16_ou_bf16 and param.__class__.__name__ != "Params4bit" and not eh_grande_demais:
            param.data = param.data.to(torch.float32)

    gc.collect()
    torch.cuda.empty_cache()

    modelo.enable_input_require_grads()
    modelo.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gradient_checkpointing_kwargs)
    return modelo


model = preparar_modelo_para_treino_kbit(model)

peft_config = LoraConfig(
    r=RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
    use_dora=USE_DORA,
)

print(f"Modelo carregado em 4 bits. LoRA rank={{RANK}}, alpha={{LORA_ALPHA}}, dora={{USE_DORA}}")

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

Memória GPU alocada antes do prepare_model_for_kbit_training: 6.74 GB
Maiores parâmetros ainda em 16 bits (candidatos ao upcast que pode estourar memória):
  model.language_model.embed_tokens_per_layer.weight: 2,348,810,240 elementos, torch.bfloat16, ~9.40 GB se virar float32
  model.language_model.embed_tokens.weight: 402,653,184 elementos, torch.bfloat16, ~1.61 GB se virar float32
  model.vision_tower.patch_embedder.position_embedding_table: 15,728,640 elementos, torch.bfloat16, ~0.06 GB se virar float32
  model.audio_tower.subsample_conv_projection.layer1.conv.weight: 36,864 elementos, torch.bfloat16, ~0.00 GB se virar float32
  model.audio_tower.layers.0.lconv1d.depthwise_conv1d.weight: 5,120 elementos, torch.bfloat16, ~0.00 GB se virar float32
Modelo carregado em 4 bits. LoRA rank={RANK}, alpha={LORA_ALPHA}, dora={USE_DORA}


**Se ainda estourar memória mesmo com o limiar acima**: aumente o `LIMIAR_ELEMENTOS_UPCAST` pra excluir também `embed_tokens.weight` (402 milhões de elementos, ~1,6GB de upcast) - já basta baixar o limiar pra algo como `100_000_000`. Se mesmo assim não resolver, o atalho mais garantido é trocar de GPU: **Runtime > Change runtime type**, escolher **A100** ou **L4** em vez de T4 (no Colab Pro costuma estar disponível), e Runtime > Run all de novo.

## Passo 4 - Treinar de verdade

`SFTTrainer` (biblioteca `trl`) é o treinador padrão da comunidade Hugging Face pra fine-tuning supervisionado - o equivalente, em Python, ao `mlx_lm.lora` do Módulo 4.2. `MAX_STEPS=20` mantém o mesmo orçamento de treino do job real do Módulo 4.2, curto de propósito: o objetivo é comparar abordagem, não treinar até convergência.

In [5]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

dataset_treino = Dataset.from_list(exemplos_treino)
dataset_validacao = Dataset.from_list(exemplos_validacao)


def collate_fn(exemplos):
    textos = [
        processor.apply_chat_template(ex["messages"], add_generation_prompt=False, tokenize=False).strip()
        for ex in exemplos
    ]
    lote = processor(text=textos, return_tensors="pt", padding=True)
    rotulos = lote["input_ids"].clone()
    rotulos[rotulos == processor.tokenizer.pad_token_id] = -100
    lote["labels"] = rotulos
    return lote


args = SFTConfig(
    output_dir="gemma-amplitude-lora-colab",
    max_length=512,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    optim="adamw_torch_fused",
    logging_steps=5,
    save_strategy="no",
    eval_strategy="steps",
    eval_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    bf16=(torch_dtype == torch.bfloat16),
    lr_scheduler_type="constant",
    report_to="none",
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    seed=0,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset_treino,
    eval_dataset=dataset_validacao,
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

resultado = trainer.train()
print(resultado)

ADAPTER_DIR = "gemma-amplitude-lora-colab-adapter"
trainer.save_model(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print(f"Adapter salvo em {ADAPTER_DIR}/ -- este é o checkpoint que a célula de inferência abaixo recarrega do zero, simulando um processo novo.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,0.949797,0.830268,0.900846,3027.000000,0.795100


TrainOutput(global_step=20, training_loss=2.0497458934783936, metrics={'train_runtime': 64.7032, 'train_samples_per_second': 0.309, 'train_steps_per_second': 0.309, 'total_flos': 43076531055168.0, 'train_loss': 2.0497458934783936, 'epoch': 0.12738853503184713})
Adapter salvo em gemma-amplitude-lora-colab-adapter/ -- este é o checkpoint que a célula de inferência abaixo recarrega do zero, simulando um processo novo.


## Passo 5 - Recarregar o adapter salvo e reproduzir o contrato stdin/stdout do Módulo 6.2

`chamar_modelo_local.py` real não treina nada: ele só carrega um checkpoint já pronto
(o adapter LoRA rank 8 do Módulo 4.2) e responde uma chamada de cada vez. A célula abaixo
reproduz exatamente isso: descarrega o modelo treinado da célula anterior, e recarrega o
adapter salvo do zero a partir do disco, simulando o que aconteceria se este fosse um
processo novo, chamado via subprocess por um orquestrador em JavaScript, do jeito que
`amplitude-seguros-assistente.js` já faz no Módulo 6.2 com a versão MLX.

In [6]:
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForMultimodalLM, AutoProcessor

# Descarrega o modelo treinado da célula anterior, pra simular um processo novo de verdade
# (o mesmo cenário real de chamar_modelo_local.py, chamado do zero a cada subprocess).
del model, trainer
gc.collect()
torch.cuda.empty_cache()

_modelo_cache = None


def carregar_modelo():
    global _modelo_cache
    if _modelo_cache is None:
        base = AutoModelForMultimodalLM.from_pretrained(
            MODEL_ID, dtype=torch_dtype, device_map="auto", quantization_config=quantization_config,
        )
        modelo = PeftModel.from_pretrained(base, ADAPTER_DIR)
        proc = AutoProcessor.from_pretrained(ADAPTER_DIR)
        _modelo_cache = (modelo, proc)
    return _modelo_cache


def chamar_modelo_local(instrucao, entrada):
    # Mesmo contrato de chamar_modelo_local.py real (Módulo 6.2): recebe instrucao+entrada,
    # devolve o texto bruto da resposta do modelo -- uma string, sem parsing.
    modelo, proc = carregar_modelo()
    texto_usuario = f"{instrucao}\n\n{entrada}"
    mensagens = [{"role": "user", "content": texto_usuario}]
    prompt = proc.apply_chat_template(mensagens, add_generation_prompt=True, tokenize=False)
    entradas = proc(text=prompt, return_tensors="pt").to(modelo.device)
    saida = modelo.generate(**entradas, max_new_tokens=150, do_sample=False)
    return proc.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True)


print("Contrato pronto: chamar_modelo_local(instrucao, entrada) -> texto bruto da resposta.")


Contrato pronto: chamar_modelo_local(instrucao, entrada) -> texto bruto da resposta.


## Passo 6 - Testar contra o mesmo exemplo real do vídeo do Módulo 6.2

Mesmo texto corrido, mesmo domínio (`amplitude-auto`), mesma instrução -- o exemplo real
mostrado ao vivo no Demo, parte 4 do Módulo 6.2, rodado ali contra os dois caminhos, nuvem
(Vertex AI) e local (MLX, Apple Silicon). Os dois bateram: segurado Carlos Eduardo Matos
Silva, placa QWE-4521, valor R$ 1.870,50. Este é o mesmo teste, terceiro caminho: HF/PEFT
via Colab.

In [7]:
INSTRUCAO = "Extraia segurado, placa e valor do orçamento de oficina abaixo."
ENTRADA = (
    "Bom dia, segue o orçamento da oficina Rio Bonito pro veículo do segurado Carlos "
    "Eduardo Matos Silva, placa QWE-4521, conserto do para-lama. As peças ficaram em R$ "
    "1.200,00 e a mão de obra em R$ 670,50, valor total do orçamento R$ 1.870,50."
)

resposta = chamar_modelo_local(INSTRUCAO, ENTRADA)
print("Resposta real (HF/PEFT via Colab):", resposta)
print()
print("Referência já verificada no vídeo do Módulo 6.2 (Vertex AI e MLX local, os dois "
      "bateram): segurado Carlos Eduardo Matos Silva, placa QWE-4521, valor 1870.5")


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Resposta real (HF/PEFT via Colab): {"segurado":"Carlos Eduardo Matos Silva","placa":"QWE-4521","valor":187050}

Referência já verificada no vídeo do Módulo 6.2 (Vertex AI e MLX local, os dois bateram): segurado Carlos Eduardo Matos Silva, placa QWE-4521, valor 1870.5


## Fechamento

Depois de rodar este notebook com GPU real: se a resposta do Passo 6 bater com a referência
(mesmo segurado, mesma placa, mesmo valor), são três caminhos independentes
(Vertex AI na nuvem, MLX em Apple Silicon local, e HF/PEFT em qualquer GPU CUDA, inclusive a
T4 grátis do Colab) convergindo pro mesmo resultado no mesmo texto nunca visto no treino.

## Levando pra fora do Colab

Numa máquina Windows ou Linux com GPU CUDA de verdade (não Colab), o mesmo código dos
Passos 5-6 vira um script standalone com o contrato exato de `chamar_modelo_local.py`: stdin
recebe `{instrucao, entrada}` em JSON, stdout devolve o texto da resposta. Ponteiro pronto:
`chamar-modelo-local-hf.py`, nesta mesma pasta, só requer um adapter já treinado e salvo em
disco (o `ADAPTER_DIR` deste notebook, baixado do Colab, resolve). Pra usar de verdade com o
`amplitude-seguros-assistente.js` do Módulo 6.2, troque o nome do script na variável `scriptLocal` dentro da função
`chamarModeloLocal` (linha 88) de `chamar_modelo_local.py` pra `chamar-modelo-local-hf.py`.

---

Ahirton Lopes · Fine-Tuning Toolkit - UNIPDS: Processamento de Dados e Fine-Tuning de Modelos
Prof. Ahirton Lopes, Ph.D. - GDE AI, Microsoft MVP, Senior Manager

## Verificação extra - o erro do Passo 6 é sistemático ou um lapso pontual?

Uma rodada anterior deste notebook errou o campo `valor` no teste do Passo 6 -
`segurado` e `placa` bateram, mas `valor` saiu `187050` em vez de `1870.5` (parece ter
concatenado os dígitos de "R$ 1.870,50" sem tratar o separador decimal brasileiro certo).
Essa célula treina um adapter LoRA do zero 3 vezes (mesma receita exata do Passo 3-4, mesmo
`seed=0`), e testa cada um contra o mesmo exemplo do Passo 6 - GPU não é perfeitamente
determinística mesmo com seed fixo (kernels do cuDNN, soma atômica no backward), então
repetir a mesma receita 3 vezes é o jeito mais direto de ver se esse erro se repete ou foi
sorte ruim de uma rodada só.

In [8]:
import gc
import torch

# Limpeza real, necessária antes de começar: os Passos 5-6 deixaram um modelo carregado
# em GPU (_modelo_cache, dentro de carregar_modelo()) e nunca liberado -- rodar 3 treinos
# novos em cima disso estourou memória na primeira rodada (achado real). Libera
# esse modelo residual antes de repetir o ciclo treino->teste.
if '_modelo_cache' in globals() and _modelo_cache is not None:
    del _modelo_cache
_modelo_cache = None
gc.collect()
torch.cuda.empty_cache()
print(f"Memória GPU alocada após limpar o modelo dos Passos 5-6: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

RESULTADOS_REPETICAO = []

for rodada in range(1, 4):
    print(f"\n===== Rodada {rodada}/3 =====")

    modelo_rodada = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, dtype=torch_dtype, device_map="auto", quantization_config=quantization_config,
    )
    modelo_rodada = preparar_modelo_para_treino_kbit(modelo_rodada)

    peft_config_rodada = LoraConfig(
        r=RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
        use_dora=USE_DORA,
    )

    args_rodada = SFTConfig(
        output_dir=f"gemma-amplitude-lora-colab-rodada{rodada}",
        max_length=512,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        optim="adamw_torch_fused",
        logging_steps=5,
        save_strategy="no",
        eval_strategy="steps",
        eval_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        bf16=(torch_dtype == torch.bfloat16),
        lr_scheduler_type="constant",
        report_to="none",
        dataset_kwargs={"skip_prepare_dataset": True},
        remove_unused_columns=False,
        seed=0,  # mesma semente das rodadas anteriores, de propósito -- queremos ver se a
                 # mesma receita reproduz o mesmo erro, não introduzir uma variável nova
    )

    trainer_rodada = SFTTrainer(
        model=modelo_rodada,
        args=args_rodada,
        train_dataset=Dataset.from_list(exemplos_treino),
        eval_dataset=Dataset.from_list(exemplos_validacao),
        peft_config=peft_config_rodada,
        processing_class=processor,
        data_collator=collate_fn,
    )
    resultado_treino = trainer_rodada.train()
    val_loss_final = resultado_treino.metrics.get("eval_loss")
    if val_loss_final is None and trainer_rodada.state.log_history:
        for entrada_log in reversed(trainer_rodada.state.log_history):
            if "eval_loss" in entrada_log:
                val_loss_final = entrada_log["eval_loss"]
                break

    prompt = processor.apply_chat_template(
        [{"role": "user", "content": f"{INSTRUCAO}\n\n{ENTRADA}"}],
        add_generation_prompt=True,
        tokenize=False,
    )
    entradas = processor(text=prompt, return_tensors="pt").to(modelo_rodada.device)
    saida = modelo_rodada.generate(**entradas, max_new_tokens=150, do_sample=False)
    resposta = processor.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True)

    print(f"Rodada {rodada}: val_loss={val_loss_final}")
    print(f"Rodada {rodada}: resposta = {resposta}")
    RESULTADOS_REPETICAO.append({"rodada": rodada, "val_loss": val_loss_final, "resposta": resposta})

    del modelo_rodada, trainer_rodada
    gc.collect()
    torch.cuda.empty_cache()

print("\n===== Resumo das 3 rodadas =====")
for r in RESULTADOS_REPETICAO:
    print(f"Rodada {r['rodada']} (val_loss={r['val_loss']}): {r['resposta']}")
print()
print("Referência (Vertex AI e MLX, já convergentes no vídeo do Módulo 6.2):")
print("segurado Carlos Eduardo Matos Silva, placa QWE-4521, valor 1870.5")


Memória GPU alocada após limpar o modelo dos Passos 5-6: 0.02 GB

===== Rodada 1/3 =====


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,0.946885,0.829984,0.874049,3027.000000,0.795952


Rodada 1: val_loss=0.8299840688705444
Rodada 1: resposta = {"segurado":"Carlos Eduardo Matos Silva","placa":"QWE-4521","valor":187050}

===== Rodada 2/3 =====


...

Rodada 2: val_loss=0.8296303153038025
Rodada 2: resposta = {"segurado":"Carlos Eduardo Matos Silva","placa":"QWE-4521","valor":187050}

===== Rodada 3/3 =====


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,0.959290,0.829630,0.910635,3027.000000,0.796522


Rodada 3: val_loss=0.8296303153038025
Rodada 3: resposta = {"segurado":"Carlos Eduardo Matos Silva","placa":"QWE-4521","valor":187050}

===== Resumo das 3 rodadas =====
Rodada 1 (val_loss=0.8299840688705444): {"segurado":"Carlos Eduardo Matos Silva","placa":"QWE-4521","valor":187050}
Rodada 2 (val_loss=0.8296303153038025): {"segurado":"Carlos Eduardo Matos Silva","placa":"QWE-4521","valor":187050}
Rodada 3 (val_loss=0.8296303153038025): {"segurado":"Carlos Eduardo Matos Silva","placa":"QWE-4521","valor":187050}

Referência (Vertex AI e MLX, já convergentes no vídeo do Módulo 6.2):
segurado Carlos Eduardo Matos Silva, placa QWE-4521, valor 1870.5


## Diagnóstico extra - o que exatamente quebra?

O erro do Passo 6 (segurado e placa certos, valor virando `187050` em vez de `1870.5`)
foi confirmado sistemático nas 3 rodadas da célula anterior. Antes de mudar qualquer
coisa da receita, vale isolar QUAL parte do valor causa isso, sem retreinar nada -
só reaproveitando o mesmo adapter já treinado no Passo 4 (`chamar_modelo_local`
recarrega ele do zero, do disco, do jeito que já faz nos Passos 5-6).

Três variações do mesmo texto, cada uma mudando uma coisa só:

- **Teste A** - mesmo texto, mesmo valor, só troca `1.870,50` por `1.870,00`
  (centavos redondos). Se acertar, o problema é especificamente o centavo
  fracionário, não o separador de milhar.
- **Teste B** - mesmo texto, valor pequeno com centavo fracionário mas SEM
  separador de milhar (`87,50`). Se acertar, o problema é a combinação
  milhar + decimal, não decimal sozinho.
- **Teste C** - um exemplo REAL do próprio dataset de treino (`Vinicius Augusto
  Teixeira`, `R$ 3.780,90`, no formato exato de nota que apareceu no treino,
  não no texto corrido do Passo 6). Se errar até esse, o modelo nunca aprendeu
  o padrão de jeito nenhum, nem pra exemplo visto. Se acertar, é generalização
  pra exemplo novo que falha, não falta de aprendizado.

In [9]:
testes_diagnostico = [
    {
        "nome": "Teste A - mesmo caso, centavos redondos (1.870,00)",
        "instrucao": "Extraia segurado, placa e valor do orçamento de oficina abaixo.",
        "entrada": (
            "Bom dia, segue o orçamento da oficina Rio Bonito pro veículo do segurado Carlos "
            "Eduardo Matos Silva, placa QWE-4521, conserto do para-lama, valor total R$ 1.870,00."
        ),
        "esperado": "valor 1870 ou 1870.0",
    },
    {
        "nome": "Teste B - valor pequeno, sem separador de milhar (87,50)",
        "instrucao": "Extraia segurado, placa e valor do orçamento de oficina abaixo.",
        "entrada": (
            "Bom dia, segue o orçamento da oficina Rio Bonito pro veículo do segurado Carlos "
            "Eduardo Matos Silva, placa QWE-4521, conserto do para-lama, valor total R$ 87,50."
        ),
        "esperado": "valor 87.5",
    },
    {
        "nome": "Teste C - exemplo real do treino (Vinicius Augusto Teixeira, formato de nota)",
        "instrucao": "Extraia segurado, placa e valor do orçamento de oficina abaixo.",
        "entrada": (
            "ATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das "
            "Turbinas 450 Distrito Industrial Segurado: Vinicius Augusto Teixeira Placa do "
            "veiculo: YHN-3392 Data do sinistro: 20/02/2026 Descricao do servico: reparo de "
            "lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.780,90"
        ),
        "esperado": "valor 3780.9 (visto no treino, exemplos_treino[0])",
    },
]

for teste in testes_diagnostico:
    resposta = chamar_modelo_local(teste["instrucao"], teste["entrada"])
    print(f"{teste['nome']}")
    print(f"  esperado: {teste['esperado']}")
    print(f"  resposta: {resposta}")
    print()


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Teste A - mesmo caso, centavos redondos (1.870,00)
  esperado: valor 1870 ou 1870.0
  resposta: {"segurado":"Carlos Eduardo Matos Silva","placa":"QWE-4521","valor":1870}

Teste B - valor pequeno, sem separador de milhar (87,50)
  esperado: valor 87.5
  resposta: {"segurado":"Carlos Eduardo Matos Silva","placa":"QWE-4521","valor":8750}

Teste C - exemplo real do treino (Vinicius Augusto Teixeira, formato de nota)
  esperado: valor 3780.9 (visto no treino, exemplos_treino[0])
  resposta: {"segurado":"Vinicius Augusto Teixeira","placa":"YHN-3392","valor":3780.9}

